# Adaptive Pullback Momentum v1.0 — Backtest

Replicates the Pine Script strategy logic bar-by-bar.

### Strategy Summary
| Component | Value |
|---|---|
| **Entry trigger** | Prev-bar low tagged EMA21 (±0.3%), current bar recovers above it bullishly |
| **Macro filter** | close > EMA200 longs / close < EMA200 shorts |
| **Structure** | EMA21 > EMA50 longs / EMA21 < EMA50 shorts |
| **Momentum** | RSI 42–68 longs / 32–58 shorts |
| **Volume** | ≥ 1.0× VolSMA 20 |
| **Trend quality** | ADX > 20 |
| **Panic filter** | ATR14 < ATR60-SMA × 1.5 |
| **Stop loss** | Entry ± ATR × 1.5 |
| **Take profit** | Entry ± ATR × 2.5 (1.67:1 R:R) |
| **Trail** | Activates at ATR×1.0 profit → trails ATR×0.8 from best price |
| **Sizing** | 1% equity at risk per trade |
| **Capital** | $10,000 · Commission 0.06% |

In [31]:
# ── 0. Install dependencies ──────────────────────────────────────────────────
import subprocess, sys
for pkg in ["yfinance", "pandas", "numpy", "matplotlib"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings("ignore")
print("Libraries ready")


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


Libraries ready



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## 1 · Configuration
Change these values to test any instrument or timeframe.

In [32]:
# ── 1. Configuration ─────────────────────────────────────────────────────────
# ▸ Data
TICKER   = "BTC-USD"
PERIOD   = "60d"      # yfinance period string  (e.g. "60d", "180d", "1y")
INTERVAL = "30m"      # "1m","5m","15m","30m","1h","1d"

# ▸ Trend filters
EMA_FAST    = 21
EMA_MID     = 50
EMA_SLOW    = 200
ADX_THRESH  = 20
ADX_LEN     = 14
PB_PCT      = 0.30    # pullback tolerance in % (how close low must come to EMA_FAST)

# ▸ Momentum / volume
RSI_LEN     = 14
RSI_LO_L    = 42;  RSI_HI_L = 68   # RSI bounds for longs
RSI_LO_S    = 32;  RSI_HI_S = 58   # RSI bounds for shorts
VOL_LEN     = 20
VOL_MULT    = 1.0

# ▸ Volatility
ATR_LEN     = 14
PANIC_MULT  = 1.5    # ATR14 > ATR60SMA × this → no new entries

# ▸ Risk management
SL_MULT     = 1.5    # stop   = entry ± ATR × SL_MULT
TP_MULT     = 2.5    # target = entry ± ATR × TP_MULT  (1.67:1 R:R)
TRAIL_ACT   = 1.0    # trail activates once price moves ATR × TRAIL_ACT in favour
TRAIL_DIST  = 0.8    # once active, trail stays ATR × TRAIL_DIST from best price
RISK_PCT    = 0.01   # fraction of equity risked per trade

# ▸ Simulation
INITIAL_CAPITAL = 10_000.0
COMMISSION_PCT  = 0.0006   # 0.06 % per side

# ▸ Direction
TRADE_LONGS  = True
TRADE_SHORTS = True

print("Config loaded.")

Config loaded.


## 2 · Download Data

In [33]:
# ── 2. Download data ─────────────────────────────────────────────────────────
df = yf.download(TICKER, period=PERIOD, interval=INTERVAL, auto_adjust=True, progress=False)

if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.droplevel(1)

df = df[["Open", "High", "Low", "Close", "Volume"]].copy()
df.dropna(inplace=True)

print(f"Downloaded: {TICKER} {INTERVAL} | Rows: {len(df)}")
print(f"  Range: {df.index[0]}  →  {df.index[-1]}")
df.tail(3)

Downloaded: BTC-USD 30m | Rows: 2842
  Range: 2026-01-03 00:00:00+00:00  →  2026-03-03 04:30:00+00:00


Price,Open,High,Low,Close,Volume
Datetime,,,,,
2026-03-03 03:30:00+00:00,68405.843750,68484.265625,68319.546875,68360.085938,823939072
2026-03-03 04:00:00+00:00,68364.468750,68429.125000,68288.125000,68315.578125,298594304
2026-03-03 04:30:00+00:00,68309.632812,68368.585938,68197.156250,68197.156250,12201984


## 3 · Indicator Calculation

In [34]:
# ── 3. Indicators ────────────────────────────────────────────────────────────

# EMA stack
df["EMA_FAST"] = df["Close"].ewm(span=EMA_FAST, adjust=False).mean()
df["EMA_MID"]  = df["Close"].ewm(span=EMA_MID,  adjust=False).mean()
df["EMA_SLOW"] = df["Close"].ewm(span=EMA_SLOW, adjust=False).mean()

# RSI (Wilder)
delta  = df["Close"].diff()
gain   = delta.clip(lower=0)
loss   = (-delta).clip(lower=0)
avg_g  = gain.ewm(alpha=1/RSI_LEN, adjust=False).mean()
avg_l  = loss.ewm(alpha=1/RSI_LEN, adjust=False).mean()
df["RSI"] = 100 - (100 / (1 + avg_g / avg_l.replace(0, 1e-10)))

# ATR (Wilder)
hl   = df["High"] - df["Low"]
hpc  = (df["High"] - df["Close"].shift(1)).abs()
lpc  = (df["Low"]  - df["Close"].shift(1)).abs()
tr   = pd.concat([hl, hpc, lpc], axis=1).max(axis=1)
df["ATR"]    = tr.ewm(alpha=1/ATR_LEN, adjust=False).mean()
df["ATR_BL"] = df["ATR"].rolling(60).mean()   # 60-bar ATR baseline

# Volume SMA
df["VOL_MA"] = df["Volume"].rolling(VOL_LEN).mean()

# ADX / DMI (Wilder)
up_move  = df["High"] - df["High"].shift(1)
dn_move  = df["Low"].shift(1) - df["Low"]
plus_dm  = np.where((up_move > dn_move) & (up_move > 0), up_move, 0.0)
minus_dm = np.where((dn_move > up_move) & (dn_move > 0), dn_move, 0.0)
s_plus   = pd.Series(plus_dm,  index=df.index).ewm(alpha=1/ADX_LEN, adjust=False).mean()
s_minus  = pd.Series(minus_dm, index=df.index).ewm(alpha=1/ADX_LEN, adjust=False).mean()
df["DI_PLUS"]  = 100 * s_plus  / df["ATR"]
df["DI_MINUS"] = 100 * s_minus / df["ATR"]
dx             = 100 * (df["DI_PLUS"] - df["DI_MINUS"]).abs() / (df["DI_PLUS"] + df["DI_MINUS"])
df["ADX"]      = dx.ewm(alpha=1/ADX_LEN, adjust=False).mean()

df.dropna(inplace=True)

# Regime flags
df["IS_TRENDING"] = df["ADX"] > ADX_THRESH
df["IS_PANIC"]    = df["ATR"] > df["ATR_BL"] * PANIC_MULT

print(f"Indicators ready. Clean rows: {len(df)}")
df[["Close","EMA_FAST","EMA_MID","EMA_SLOW","RSI","ATR","ADX","IS_TRENDING","IS_PANIC"]].tail(4)

Indicators ready. Clean rows: 2783


Price,Close,EMA_FAST,EMA_MID,EMA_SLOW,RSI,ATR,ADX,IS_TRENDING,IS_PANIC
Datetime,,,,,,,,,
2026-03-03 03:00:00+00:00,68389.562500,68640.412874,67963.177543,66881.535678,47.972184,380.183601,23.522222,True,False
2026-03-03 03:30:00+00:00,68360.085938,68614.928607,67978.742578,66896.247621,47.428438,364.793254,22.479941,True,False
2026-03-03 04:00:00+00:00,68315.578125,68587.714927,67991.951815,66910.370313,46.570102,348.808022,21.594644,True,False
2026-03-03 04:30:00+00:00,68197.156250,68552.209593,67999.999048,66923.174153,44.274218,336.138141,21.016975,True,False


## 4 · Entry Signals

In [35]:
# ── 4. Entry signals ─────────────────────────────────────────────────────────
tol = PB_PCT / 100.0

# Long  : prev bar's low ≤ EMA_FAST × (1 + tol)  AND  current close > EMA_FAST, close > open
long_pb  = (
    (df["Low"].shift(1) <= df["EMA_FAST"].shift(1) * (1 + tol)) &
    (df["Close"] > df["EMA_FAST"]) &
    (df["Close"] > df["Open"])
)

# Short : prev bar's high ≥ EMA_FAST × (1 - tol)  AND  current close < EMA_FAST, close < open
short_pb = (
    (df["High"].shift(1) >= df["EMA_FAST"].shift(1) * (1 - tol)) &
    (df["Close"] < df["EMA_FAST"]) &
    (df["Close"] < df["Open"])
)

df["LongSignal"] = (
    long_pb &
    (df["Close"]    > df["EMA_SLOW"]) &
    (df["EMA_FAST"] > df["EMA_MID"]) &
    (df["RSI"] >= RSI_LO_L) & (df["RSI"] <= RSI_HI_L) &
    (df["Volume"] >= df["VOL_MA"] * VOL_MULT) &
    df["IS_TRENDING"] & ~df["IS_PANIC"]
) & TRADE_LONGS

df["ShortSignal"] = (
    short_pb &
    (df["Close"]    < df["EMA_SLOW"]) &
    (df["EMA_FAST"] < df["EMA_MID"]) &
    (df["RSI"] >= RSI_LO_S) & (df["RSI"] <= RSI_HI_S) &
    (df["Volume"] >= df["VOL_MA"] * VOL_MULT) &
    df["IS_TRENDING"] & ~df["IS_PANIC"]
) & TRADE_SHORTS

print(f"Long  signals : {df['LongSignal'].sum():>4}")
print(f"Short signals : {df['ShortSignal'].sum():>4}")
print(f"Total         : {df['LongSignal'].sum() + df['ShortSignal'].sum():>4}")

Long  signals :   28
Short signals :   66
Total         :   94


## 5 · Backtest Simulation

In [36]:
# ── 5. Bar-by-bar backtest ────────────────────────────────────────────────────
equity       = INITIAL_CAPITAL
position     = None
trades       = []
equity_curve = []

for ts, row in df.iterrows():
    close = float(row["Close"])
    high  = float(row["High"])
    low   = float(row["Low"])
    atr   = float(row["ATR"])
    stop_dist = atr * SL_MULT

    hit_tp = hit_sl = False
    exit_px = None

    # ── Manage open position ──────────────────────────────────────────────────
    if position is not None:
        d = position["direction"]

        if d == "long":
            if high > position["best"]:
                position["best"] = high
            # Activate / update trailing stop
            if position["best"] >= position["entry"] + atr * TRAIL_ACT:
                candidate = position["best"] - atr * TRAIL_DIST
                position["sl"] = max(position["sl"], candidate)
            hit_tp = high >= position["tp"]
            hit_sl = low  <= position["sl"]
            if hit_tp or hit_sl:
                exit_px = position["tp"] if hit_tp else position["sl"]
                pnl_pct = (exit_px - position["entry"]) / position["entry"]

        else:  # short
            if low < position["best"]:
                position["best"] = low
            if position["best"] <= position["entry"] - atr * TRAIL_ACT:
                candidate = position["best"] + atr * TRAIL_DIST
                position["sl"] = min(position["sl"], candidate)
            hit_tp = low  <= position["tp"]
            hit_sl = high >= position["sl"]
            if hit_tp or hit_sl:
                exit_px = position["tp"] if hit_tp else position["sl"]
                pnl_pct = (position["entry"] - exit_px) / position["entry"]

        if hit_tp or hit_sl:
            dollar_pnl  = pnl_pct * position["notional"]
            commission  = position["notional"] * COMMISSION_PCT * 2
            dollar_pnl -= commission
            equity     += dollar_pnl
            trades.append({
                "entry_time" : position["entry_time"],
                "exit_time"  : ts,
                "direction"  : d,
                "entry"      : position["entry"],
                "exit"       : exit_px,
                "sl_initial" : position["sl_initial"],
                "tp"         : position["tp"],
                "result"     : "TP" if hit_tp else "SL",
                "pnl_pct"    : round(pnl_pct * 100, 3),
                "dollar_pnl" : round(dollar_pnl, 2),
                "equity"     : round(equity, 2),
            })
            position = None

    # ── New entry ─────────────────────────────────────────────────────────────
    if position is None:
        sig = None
        if   bool(row["LongSignal"]):  sig = "long"
        elif bool(row["ShortSignal"]): sig = "short"

        if sig:
            risk_cap = equity * RISK_PCT
            qty      = risk_cap / stop_dist
            notional = qty * close
            sl = close - stop_dist if sig == "long" else close + stop_dist
            tp = close + atr * TP_MULT if sig == "long" else close - atr * TP_MULT
            position = {
                "direction"  : sig,
                "entry"      : close,
                "entry_time" : ts,
                "sl"         : sl,
                "sl_initial" : sl,
                "tp"         : tp,
                "best"       : close,
                "notional"   : notional,
            }

    equity_curve.append({"time": ts, "equity": equity})

print(f"Simulation complete.  Trades: {len(trades)}")

Simulation complete.  Trades: 74


## 6 · Performance Statistics

In [37]:
# ── 6. Performance statistics ────────────────────────────────────────────────
tdf = pd.DataFrame(trades)

if tdf.empty:
    print("No trades executed — try relaxing ADX, RSI bounds or pullback tolerance.")
else:
    wins   = tdf[tdf["dollar_pnl"] > 0]
    losses = tdf[tdf["dollar_pnl"] <= 0]
    wr     = len(wins) / len(tdf) * 100
    total  = tdf["dollar_pnl"].sum()
    final  = tdf["equity"].iloc[-1]
    gp     = wins["dollar_pnl"].sum()   if not wins.empty   else 0
    gl     = losses["dollar_pnl"].sum() if not losses.empty else 0
    pf     = gp / abs(gl)               if gl != 0          else float("inf")

    eq_s   = pd.Series([e["equity"] for e in equity_curve])
    mdd    = ((eq_s - eq_s.cummax()) / eq_s.cummax() * 100).min()
    ret    = (final / INITIAL_CAPITAL - 1) * 100

    tp_cnt = (tdf["result"] == "TP").sum()
    sl_cnt = (tdf["result"] == "SL").sum()
    long_c = (tdf["direction"] == "long").sum()
    shrt_c = (tdf["direction"] == "short").sum()

    avg_win  = wins["dollar_pnl"].mean()   if not wins.empty   else 0
    avg_loss = losses["dollar_pnl"].mean() if not losses.empty else 0
    rr       = avg_win / abs(avg_loss)      if avg_loss != 0    else float("inf")

    print("=" * 56)
    print(f"  APM v1.0  —  {TICKER} {INTERVAL}  ({PERIOD})")
    print("=" * 56)
    print(f"  Initial capital   : ${INITIAL_CAPITAL:>10,.2f}")
    print(f"  Final equity      : ${final:>10,.2f}")
    print(f"  Net P&L           : ${total:>+10,.2f}")
    print(f"  Return            : {ret:>+9.2f} %")
    print(f"  Max drawdown      : {mdd:>9.2f} %")
    print(f"  Profit factor     : {pf:>9.2f}")
    print("-" * 56)
    print(f"  Total trades      : {len(tdf):>5}")
    print(f"    Long  trades    : {long_c:>5}")
    print(f"    Short trades    : {shrt_c:>5}")
    print(f"  TP exits          : {tp_cnt:>5}")
    print(f"  SL exits          : {sl_cnt:>5}")
    print(f"  Win rate          : {wr:>8.1f} %")
    print(f"  Avg win  $        : ${avg_win:>+9,.2f}")
    print(f"  Avg loss $        : ${avg_loss:>+9,.2f}")
    print(f"  Avg R:R           : {rr:>9.2f}")
    print(f"  Best trade  $     : ${tdf['dollar_pnl'].max():>+9,.2f}")
    print(f"  Worst trade $     : ${tdf['dollar_pnl'].min():>+9,.2f}")
    print("=" * 56)

tdf

  APM v1.0  —  BTC-USD 30m  (60d)
  Initial capital   : $ 10,000.00
  Final equity      : $  7,968.47
  Net P&L           : $ -2,031.58
  Return            :    -20.32 %
  Max drawdown      :    -20.75 %
  Profit factor     :      0.39
--------------------------------------------------------
  Total trades      :    74
    Long  trades    :    23
    Short trades    :    51
  TP exits          :     5
  SL exits          :    69
  Win rate          :     48.6 %
  Avg win  $        : $   +36.77
  Avg loss $        : $   -88.29
  Avg R:R           :      0.42
  Best trade  $     : $  +134.70
  Worst trade $     : $  -158.77


,entry_time,exit_time,direction,entry,exit,sl_initial,tp,result,pnl_pct,dollar_pnl,equity
0,2026-01-04 07:00:00+00:00,2026-01-04 07:30:00+00:00,long,91251.617188,91312.440981,90949.266945,91755.534258,SL,0.067,-16.10,9983.90
1,2026-01-04 10:00:00+00:00,2026-01-04 12:30:00+00:00,long,91440.375000,91146.452077,91146.452077,91930.246538,SL,-0.321,-137.11,9846.79
2,2026-01-04 20:00:00+00:00,2026-01-05 00:00:00+00:00,long,91289.000000,91726.639176,91026.416494,91726.639176,TP,0.479,123.03,9969.82
3,2026-01-05 09:00:00+00:00,2026-01-05 10:00:00+00:00,long,92541.851562,92681.882547,92171.528846,93159.056090,SL,0.151,7.80,9977.62
4,2026-01-07 14:00:00+00:00,2026-01-07 14:30:00+00:00,short,91470.875000,91900.360013,91900.360013,90755.066646,SL,-0.470,-125.28,9852.35
...,...,...,...,...,...,...,...,...,...,...,...
69,2026-02-26 05:30:00+00:00,2026-02-26 14:00:00+00:00,long,68429.953125,67838.187614,67838.187614,69416.228977,SL,-0.865,-92.28,8010.91
70,2026-02-28 01:30:00+00:00,2026-02-28 05:00:00+00:00,short,65811.437500,65688.773679,66230.565081,65112.891532,SL,0.186,8.35,8019.26
71,2026-03-01 01:30:00+00:00,2026-03-01 02:00:00+00:00,long,67271.445312,67719.931806,66517.439525,68528.121625,SL,0.667,39.11,8058.37
72,2026-03-01 02:00:00+00:00,2026-03-01 05:30:00+00:00,long,67686.546875,67783.196663,66893.001043,69009.123261,SL,0.143,1.57,8059.94


## 7 · Charts

In [38]:
# ── 7. Charts ─────────────────────────────────────────────────────────────────
ec_df = pd.DataFrame(equity_curve).set_index("time")
plt.style.use("dark_background")

fig, axes = plt.subplots(4, 1, figsize=(18, 18),
                          gridspec_kw={"height_ratios": [3, 1, 1.5, 1.5]})
fig.suptitle(
    f"APM v1.0  ·  {TICKER} {INTERVAL}  ·  Pull-to-EMA{EMA_FAST} | ADX>{ADX_THRESH} | RSI {RSI_LO_L}–{RSI_HI_L} | "
    f"SL×{SL_MULT} / TP×{TP_MULT} / Trail×{TRAIL_DIST}",
    fontsize=12)

# ── Panel 1 : Price + EMAs + entry/exit markers ───────────────────────────────
ax1 = axes[0]
ax1.plot(df.index, df["Close"],    color="#cccccc", lw=0.7, label="Close")
ax1.plot(df.index, df["EMA_SLOW"], color="#f6e05e", lw=2.0, label=f"EMA {EMA_SLOW}")
ax1.plot(df.index, df["EMA_MID"],  color="#f6ad55", lw=1.0, ls="--", alpha=0.8, label=f"EMA {EMA_MID}")
ax1.plot(df.index, df["EMA_FAST"], color="#5b9ef4", lw=1.0, ls="--", alpha=0.8, label=f"EMA {EMA_FAST}")

if not tdf.empty:
    for _, t in tdf.iterrows():
        w_col = "#68d391" if t["dollar_pnl"] >= 0 else "#fc8181"
        mrkr  = "^" if t["direction"] == "long" else "v"
        e_col = "#68d391" if t["direction"] == "long" else "#fc8181"
        ax1.scatter(t["entry_time"], t["entry"], marker=mrkr, color=e_col, s=70, zorder=5)
        ax1.scatter(t["exit_time"],  t["exit"],  marker="x",  color=w_col, s=50, zorder=5)

ax1.set_ylabel("Price (USD)")
ax1.legend(loc="upper left", fontsize=8)
ax1.grid(alpha=0.15)
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))

# ── Panel 2 : ADX + RSI (dual axis) + panic shading ──────────────────────────
ax2     = axes[1]
ax2_rsi = ax2.twinx()
ax2.plot(df.index, df["ADX"], color="#a0aec0", lw=1.0, label=f"ADX {ADX_LEN}")
ax2.axhline(ADX_THRESH, color="#68d391", ls="--", lw=0.8, alpha=0.7, label=f"ADX={ADX_THRESH}")
ax2_rsi.plot(df.index, df["RSI"], color="#f6ad55", lw=0.9, alpha=0.7, label="RSI")
ax2_rsi.axhline(RSI_HI_L, color="#fc8181", ls=":", lw=0.7, alpha=0.6)
ax2_rsi.axhline(RSI_LO_L, color="#68d391", ls=":", lw=0.7, alpha=0.6)
ax2_rsi.set_ylim(0, 100)
for ts_p in df[df["IS_PANIC"]].index:
    ax2.axvspan(ts_p, ts_p, alpha=0.08, color="red")
ax2.set_ylabel("ADX",  color="#a0aec0")
ax2_rsi.set_ylabel("RSI", color="#f6ad55")
ax2.legend(loc="upper left", fontsize=7)
ax2.grid(alpha=0.15)
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))

# ── Panel 3 : Equity curve ────────────────────────────────────────────────────
ax3 = axes[2]
final_val  = ec_df["equity"].iloc[-1]
eq_col     = "#68d391" if final_val >= INITIAL_CAPITAL else "#fc8181"
ax3.plot(ec_df.index, ec_df["equity"], color=eq_col, lw=1.5)
ax3.axhline(INITIAL_CAPITAL, color="white", ls=":", lw=0.8, alpha=0.5)
ax3.fill_between(ec_df.index, INITIAL_CAPITAL, ec_df["equity"],
                 where=ec_df["equity"] >= INITIAL_CAPITAL, alpha=0.2, color="#68d391")
ax3.fill_between(ec_df.index, INITIAL_CAPITAL, ec_df["equity"],
                 where=ec_df["equity"] < INITIAL_CAPITAL,  alpha=0.2, color="#fc8181")
ax3.set_ylabel("Equity ($)")
ax3.grid(alpha=0.15)
ax3.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))

# ── Panel 4 : Per-trade P&L bars ─────────────────────────────────────────────
ax4 = axes[3]
if not tdf.empty:
    bar_c  = ["#68d391" if v >= 0 else "#fc8181" for v in tdf["dollar_pnl"]]
    ax4.bar(range(len(tdf)), tdf["dollar_pnl"], color=bar_c, width=0.6)
    ax4.axhline(0, color="white", lw=0.7, alpha=0.5)
    labels = [f"T{i+1}\n{r}" for i, r in enumerate(tdf["result"])]
    ax4.set_xticks(range(len(tdf)))
    ax4.set_xticklabels(labels, fontsize=7)
ax4.set_ylabel("P&L ($)")
ax4.grid(alpha=0.15)

plt.tight_layout()
out_png = f"backtest_apm_{TICKER.replace('-','').lower()}_{INTERVAL}.png"
plt.savefig(out_png, dpi=150, bbox_inches="tight")
plt.show()
print(f"Chart saved → {out_png}")

Chart saved → backtest_apm_btcusd_30m.png


## 8 · Diagnosis & Parameter Sensitivity

**Observations from first run (BTC-USD 30m, 60d, Jan–Mar 2026)**

| Root cause | Evidence | Fix |
|---|---|---|
| TP too far → almost never hit | 5/72 TP exits; avg win only $39 vs target-implied ~$166 | Tighten TP to ATR×1.5 (1:1 R:R but rely on trail to extend good trades) |
| Trailing stop exits winners too early | Avg win $39 ≈ only 0.44× the avg loss; trail at 0.8×ATR is very tight | Raise TRAIL_ACT to 1.5 so trail activates later; test TRAIL_DIST at 1.0 |
| PB_PCT = 0.30% too permissive | 91 raw signals = 52/week — too many noise entries | Raise to 0.15% (tighter touch) or add min-body-size filter |
| No required bar-body minimum | Entry bar could be near-doji | Require `(close - open) / atr > 0.15` for body confirmation |

**Quick sensitivity test below tries tighter parameters.**

In [39]:

# ── 8. Parameter sensitivity — tighter configuration ─────────────────────────
# Changes vs defaults:
#   PB_PCT      : 0.30 → 0.15   (tighter pullback touch required)
#   TP_MULT     : 2.50 → 1.50   (closer target → more TP exits)
#   TRAIL_ACT   : 1.00 → 1.50   (trail activates later → full TP has more room)
#   TRAIL_DIST  : 0.80 → 1.00   (looser trail → fewer premature exits)
#   Min body pct: new  → 0.15×ATR  (entry bar must have meaningful body)
#   ADX_THRESH  : 20   → 25     (stricter trend quality)

PB2      = 0.15
TP2      = 1.50
TACT2    = 1.50
TDIST2   = 1.00
ADX2     = 25
MIN_BODY = 0.15   # (close-open)/atr must exceed this for entry bar confirmation

tol2 = PB2 / 100.0

# Body confirmation
body_bull = (df["Close"] - df["Open"]) / df["ATR"] > MIN_BODY
body_bear = (df["Open"] - df["Close"]) / df["ATR"] > MIN_BODY

long_pb2  = (
    (df["Low"].shift(1) <= df["EMA_FAST"].shift(1) * (1 + tol2)) &
    (df["Close"] > df["EMA_FAST"]) & body_bull
)
short_pb2 = (
    (df["High"].shift(1) >= df["EMA_FAST"].shift(1) * (1 - tol2)) &
    (df["Close"] < df["EMA_FAST"]) & body_bear
)

df["LongSignal2"] = (
    long_pb2 &
    (df["Close"]    > df["EMA_SLOW"]) &
    (df["EMA_FAST"] > df["EMA_MID"]) &
    (df["RSI"] >= RSI_LO_L) & (df["RSI"] <= RSI_HI_L) &
    (df["Volume"] >= df["VOL_MA"] * VOL_MULT) &
    (df["ADX"] > ADX2) & ~df["IS_PANIC"]
) & TRADE_LONGS

df["ShortSignal2"] = (
    short_pb2 &
    (df["Close"]    < df["EMA_SLOW"]) &
    (df["EMA_FAST"] < df["EMA_MID"]) &
    (df["RSI"] >= RSI_LO_S) & (df["RSI"] <= RSI_HI_S) &
    (df["Volume"] >= df["VOL_MA"] * VOL_MULT) &
    (df["ADX"] > ADX2) & ~df["IS_PANIC"]
) & TRADE_SHORTS

print(f"v2 Long  signals : {df['LongSignal2'].sum():>4}")
print(f"v2 Short signals : {df['ShortSignal2'].sum():>4}")
print(f"v2 Total         : {df['LongSignal2'].sum() + df['ShortSignal2'].sum():>4}")

# ── Run simulation ────────────────────────────────────────────────────────────
eq2       = INITIAL_CAPITAL
pos2      = None
trades2   = []
eqcurve2  = []

for ts, row in df.iterrows():
    close = float(row["Close"])
    high  = float(row["High"])
    low   = float(row["Low"])
    atr   = float(row["ATR"])
    sd    = atr * SL_MULT   # stop distance stays the same (1.5×ATR)

    if pos2 is not None:
        d = pos2["direction"]
        if d == "long":
            if high > pos2["best"]: pos2["best"] = high
            if pos2["best"] >= pos2["entry"] + atr * TACT2:
                pos2["sl"] = max(pos2["sl"], pos2["best"] - atr * TDIST2)
            htp = high >= pos2["tp"];  hsl = low <= pos2["sl"]
            if htp or hsl:
                xp = pos2["tp"] if htp else pos2["sl"]
                pnl = (xp - pos2["entry"]) / pos2["entry"]
        else:
            if low < pos2["best"]: pos2["best"] = low
            if pos2["best"] <= pos2["entry"] - atr * TACT2:
                pos2["sl"] = min(pos2["sl"], pos2["best"] + atr * TDIST2)
            htp = low <= pos2["tp"];  hsl = high >= pos2["sl"]
            if htp or hsl:
                xp = pos2["tp"] if htp else pos2["sl"]
                pnl = (pos2["entry"] - xp) / pos2["entry"]

        if htp or hsl:
            dpnl = pnl * pos2["notional"] - pos2["notional"] * COMMISSION_PCT * 2
            eq2 += dpnl
            trades2.append({
                "direction": d, "entry_time": pos2["entry_time"], "exit_time": ts,
                "entry": pos2["entry"], "exit": xp,
                "result": "TP" if htp else "SL",
                "pnl_pct": round(pnl*100, 3),
                "dollar_pnl": round(dpnl, 2), "equity": round(eq2, 2),
            })
            pos2 = None

    if pos2 is None:
        sig = "long" if bool(row["LongSignal2"]) else ("short" if bool(row["ShortSignal2"]) else None)
        if sig:
            rc = eq2 * RISK_PCT; qty = rc / sd; notional = qty * close
            sl = close - sd if sig == "long" else close + sd
            tp = close + atr * TP2 if sig == "long" else close - atr * TP2
            pos2 = {"direction": sig, "entry": close, "entry_time": ts,
                    "sl": sl, "tp": tp, "best": close, "notional": notional}

    eqcurve2.append({"time": ts, "equity": eq2})

t2df = pd.DataFrame(trades2)
print(f"\nv2 trades executed: {len(t2df)}")

# ── Stats comparison ──────────────────────────────────────────────────────────
def stats_block(tdf_, eqc_, label):
    if tdf_.empty:
        print(f"[{label}] No trades."); return
    w = tdf_[tdf_["dollar_pnl"] > 0]; l = tdf_[tdf_["dollar_pnl"] <= 0]
    wr  = len(w)/len(tdf_)*100
    fin = tdf_["equity"].iloc[-1]
    gp  = w["dollar_pnl"].sum()   if not w.empty else 0
    gl  = l["dollar_pnl"].sum()   if not l.empty else 0
    pf  = gp/abs(gl)              if gl != 0     else float("inf")
    mdd = ((pd.Series([e["equity"] for e in eqc_]) -
            pd.Series([e["equity"] for e in eqc_]).cummax()) /
           pd.Series([e["equity"] for e in eqc_]).cummax() * 100).min()
    tp_c = (tdf_["result"]=="TP").sum()
    aw   = w["dollar_pnl"].mean() if not w.empty else 0
    al   = l["dollar_pnl"].mean() if not l.empty else 0
    rr   = aw/abs(al)             if al != 0     else float("inf")
    ret  = (fin/INITIAL_CAPITAL-1)*100
    print(f"{'='*54}")
    print(f"  {label}")
    print(f"{'='*54}")
    print(f"  Return        : {ret:>+9.2f} %   Max DD: {mdd:.2f} %")
    print(f"  Profit factor : {pf:>9.2f}   Trades: {len(tdf_)}")
    print(f"  Win rate      : {wr:>8.1f} %   TP exits: {tp_c} / {len(tdf_)}")
    print(f"  Avg win  $    : ${aw:>+9,.2f}   Avg loss $: ${al:>+9,.2f}   R:R: {rr:.2f}")
    print(f"{'='*54}")

stats_block(tdf,   equity_curve, f"APM v1 defaults   (PB={PB_PCT}% TP×{TP_MULT} ADX>{ADX_THRESH})")
print()
stats_block(t2df, eqcurve2,    f"APM v1 tighter    (PB={PB2}%  TP×{TP2}  ADX>{ADX2} +body)")


v2 Long  signals :   12
v2 Short signals :   31
v2 Total         :   43

v2 trades executed: 38
  APM v1 defaults   (PB=0.3% TP×2.5 ADX>20)
  Return        :    -20.32 %   Max DD: -20.75 %
  Profit factor :      0.39   Trades: 74
  Win rate      :     48.6 %   TP exits: 5 / 74
  Avg win  $    : $   +36.77   Avg loss $: $   -88.29   R:R: 0.42

  APM v1 tighter    (PB=0.15%  TP×1.5  ADX>25 +body)
  Return        :     +2.71 %   Max DD: -4.44 %
  Profit factor :      1.17   Trades: 38
  Win rate      :     65.8 %   TP exits: 23 / 38
  Avg win  $    : $   +73.32   Avg loss $: $  -120.16   R:R: 0.61


## 9. Equity Curve Comparison — v1 defaults vs v2 tighter

In [40]:
# ── 9. Equity curve comparison ────────────────────────────────────────────────
ec1 = pd.DataFrame(equity_curve).set_index("time")["equity"]
ec2 = pd.DataFrame(eqcurve2).set_index("time")["equity"]

plt.style.use("dark_background")
fig, axes = plt.subplots(2, 1, figsize=(18, 10),
                          gridspec_kw={"height_ratios": [3, 1]})
fig.suptitle(f"APM v1.0  ·  {TICKER} {INTERVAL}  ·  Equity Curve Comparison", fontsize=13)

ax = axes[0]
ax.plot(ec1.index, ec1.values, color="#fc8181", lw=1.5,
        label=f"v1 defaults  (PB={PB_PCT}% TP×{TP_MULT} ADX>{ADX_THRESH})")
ax.plot(ec2.index, ec2.values, color="#68d391", lw=1.5,
        label=f"v2 tighter   (PB={PB2}%  TP×{TP2}  ADX>{ADX2} +body)")
ax.axhline(INITIAL_CAPITAL, color="white", ls=":", lw=0.8, alpha=0.5, label="Starting capital")
ax.fill_between(ec1.index, INITIAL_CAPITAL, ec1.values,
                where=ec1.values >= INITIAL_CAPITAL, alpha=0.08, color="#68d391")
ax.fill_between(ec1.index, INITIAL_CAPITAL, ec1.values,
                where=ec1.values < INITIAL_CAPITAL,  alpha=0.12, color="#fc8181")
ax.fill_between(ec2.index, INITIAL_CAPITAL, ec2.values,
                where=ec2.values >= INITIAL_CAPITAL, alpha=0.12, color="#68d391")
ax.fill_between(ec2.index, INITIAL_CAPITAL, ec2.values,
                where=ec2.values < INITIAL_CAPITAL,  alpha=0.08, color="#fc8181")
ax.set_ylabel("Equity ($)")
ax.legend(loc="lower left", fontsize=9)
ax.grid(alpha=0.15)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))

# Drawdown panel
def rolling_dd(ec):
    running_max = ec.cummax()
    return (ec - running_max) / running_max * 100

dd1 = rolling_dd(ec1)
dd2 = rolling_dd(ec2)
ax2 = axes[1]
ax2.fill_between(dd1.index, dd1.values, 0, alpha=0.4, color="#fc8181", label="v1 DD")
ax2.fill_between(dd2.index, dd2.values, 0, alpha=0.4, color="#68d391", label="v2 DD")
ax2.set_ylabel("Drawdown (%)")
ax2.legend(loc="lower left", fontsize=8)
ax2.grid(alpha=0.15)
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))

plt.tight_layout()
cmp_png = f"backtest_apm_comparison_{TICKER.replace('-','').lower()}_{INTERVAL}.png"
plt.savefig(cmp_png, dpi=150, bbox_inches="tight")
plt.show()
print(f"Chart saved → {cmp_png}")


Chart saved → backtest_apm_comparison_btcusd_30m.png


## 10. Monthly Returns Breakdown

In [41]:
# ── 10. Monthly returns breakdown ─────────────────────────────────────────────
def monthly_returns(trades_df, label):
    if trades_df.empty:
        print(f"[{label}] No trades to analyse.")
        return
    df_t = trades_df.copy()
    df_t["month"] = pd.to_datetime(df_t["exit_time"]).dt.to_period("M")
    monthly = df_t.groupby("month")["dollar_pnl"].sum()
    win_months  = (monthly > 0).sum()
    loss_months = (monthly <= 0).sum()
    print(f"\n{'─'*50}")
    print(f"  {label}")
    print(f"{'─'*50}")
    print(f"  {'Month':<12} {'P&L $':>10}  {'':>6}")
    print(f"{'─'*50}")
    for m, pnl_val in monthly.items():
        bar = "█" * min(int(abs(pnl_val) / 20), 30)
        sign = "+" if pnl_val >= 0 else ""
        col_flag = "▲" if pnl_val >= 0 else "▼"
        print(f"  {str(m):<12} {sign}{pnl_val:>9,.2f}  {col_flag}  {bar}")
    print(f"{'─'*50}")
    print(f"  Winning months: {win_months}  |  Losing months: {loss_months}")

monthly_returns(tdf,   f"v1 defaults  (PB={PB_PCT}% TP×{TP_MULT} ADX>{ADX_THRESH})")
monthly_returns(t2df,  f"v2 tighter   (PB={PB2}%  TP×{TP2}  ADX>{ADX2} +body)")

# Heat-map style monthly grid
def monthly_heatmap(ec_series, label, ax):
    ec = ec_series.copy()
    ec.index = pd.to_datetime(ec.index)
    monthly_eq = ec.resample("ME").last()
    monthly_ret = monthly_eq.pct_change() * 100
    monthly_ret = monthly_ret.dropna()
    pivot = pd.DataFrame({
        "year":  monthly_ret.index.year,
        "month": monthly_ret.index.strftime("%b"),
        "ret":   monthly_ret.values
    })
    month_order = ["Jan","Feb","Mar","Apr","May","Jun",
                   "Jul","Aug","Sep","Oct","Nov","Dec"]
    pivot["month"] = pd.Categorical(pivot["month"], categories=month_order, ordered=True)
    pivot = pivot.pivot_table(index="year", columns="month", values="ret")
    pivot = pivot.reindex(columns=month_order)
    import matplotlib.colors as mcolors
    cmap = mcolors.LinearSegmentedColormap.from_list("rg", ["#fc8181","#1a202c","#68d391"])
    vmax = max(abs(pivot.values[~np.isnan(pivot.values)]).max(), 1)
    im = ax.imshow(pivot.values, cmap=cmap, vmin=-vmax, vmax=vmax, aspect="auto")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, fontsize=8)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index, fontsize=8)
    for r in range(pivot.shape[0]):
        for c in range(pivot.shape[1]):
            v = pivot.values[r, c]
            if not np.isnan(v):
                ax.text(c, r, f"{v:+.1f}%", ha="center", va="center",
                        fontsize=7, color="white" if abs(v) < vmax * 0.5 else "black")
    ax.set_title(label, fontsize=9)
    return im

fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(18, 4))
fig.suptitle(f"Monthly Returns Heatmap  ·  {TICKER} {INTERVAL}", fontsize=12)
plt.style.use("dark_background")
monthly_heatmap(ec1, f"v1 defaults  (TP×{TP_MULT} ADX>{ADX_THRESH})", ax_a)
monthly_heatmap(ec2, f"v2 tighter   (TP×{TP2} ADX>{ADX2} +body)",    ax_b)
plt.tight_layout()
plt.savefig(f"backtest_apm_monthly_{TICKER.replace('-','').lower()}_{INTERVAL}.png", dpi=150, bbox_inches="tight")
plt.show()



──────────────────────────────────────────────────
  v1 defaults  (PB=0.3% TP×2.5 ADX>20)
──────────────────────────────────────────────────
  Month             P&L $        
──────────────────────────────────────────────────
  2026-01        -838.51  ▼  ██████████████████████████████
  2026-02      -1,142.28  ▼  ██████████████████████████████
  2026-03         -50.79  ▼  ██
──────────────────────────────────────────────────
  Winning months: 0  |  Losing months: 3

──────────────────────────────────────────────────
  v2 tighter   (PB=0.15%  TP×1.5  ADX>25 +body)
──────────────────────────────────────────────────
  Month             P&L $        
──────────────────────────────────────────────────
  2026-01      +   350.07  ▲  █████████████████
  2026-02      +    61.58  ▲  ███
  2026-03        -140.73  ▼  ███████
──────────────────────────────────────────────────
  Winning months: 2  |  Losing months: 1


## 11. Parameter Sweep — PB% × TP multiplier grid

In [42]:
# ── 11. Parameter sweep : PB_PCT × TP_MULT ────────────────────────────────────
# Grid search over pullback tolerance and take-profit multiplier.
# All other params stay at v2 "tighter" values (ADX>25, body filter, etc.)

pb_vals = [0.10, 0.15, 0.20, 0.30, 0.40]
tp_vals = [1.0,  1.5,  2.0,  2.5,  3.0]

def run_sim(df_, pb_pct_, tp_mult_,
            adx_thresh_=ADX2, min_body_=MIN_BODY,
            trail_act_=TACT2, trail_dist_=TDIST2):
    tol_ = pb_pct_ / 100.0
    bb_  = (df_["Close"] - df_["Open"]) / df_["ATR"] > min_body_
    bs_  = (df_["Open"] - df_["Close"]) / df_["ATR"] > min_body_
    lp_  = ((df_["Low"].shift(1) <= df_["EMA_FAST"].shift(1) * (1 + tol_)) &
             (df_["Close"] > df_["EMA_FAST"]) & bb_)
    sp_  = ((df_["High"].shift(1) >= df_["EMA_FAST"].shift(1) * (1 - tol_)) &
             (df_["Close"] < df_["EMA_FAST"]) & bs_)
    long_sig  = (lp_ & (df_["Close"] > df_["EMA_SLOW"]) &
                  (df_["EMA_FAST"] > df_["EMA_MID"]) &
                  (df_["RSI"] >= RSI_LO_L) & (df_["RSI"] <= RSI_HI_L) &
                  (df_["Volume"] >= df_["VOL_MA"] * VOL_MULT) &
                  (df_["ADX"] > adx_thresh_) & ~df_["IS_PANIC"]) & TRADE_LONGS
    short_sig = (sp_ & (df_["Close"] < df_["EMA_SLOW"]) &
                  (df_["EMA_FAST"] < df_["EMA_MID"]) &
                  (df_["RSI"] >= RSI_LO_S) & (df_["RSI"] <= RSI_HI_S) &
                  (df_["Volume"] >= df_["VOL_MA"] * VOL_MULT) &
                  (df_["ADX"] > adx_thresh_) & ~df_["IS_PANIC"]) & TRADE_SHORTS

    eq_, pos_, trades_ = INITIAL_CAPITAL, None, []
    for ts_, row_ in df_.iterrows():
        cl_ = float(row_["Close"]); hi_ = float(row_["High"])
        lo_ = float(row_["Low"]);   at_ = float(row_["ATR"])
        sd_ = at_ * SL_MULT
        htp_ = hsl_ = False; pnl_ = 0.0
        if pos_ is not None:
            d_ = pos_["direction"]
            if d_ == "long":
                if hi_ > pos_["best"]: pos_["best"] = hi_
                if pos_["best"] >= pos_["entry"] + at_ * trail_act_:
                    pos_["sl"] = max(pos_["sl"], pos_["best"] - at_ * trail_dist_)
                htp_ = hi_ >= pos_["tp"]; hsl_ = lo_ <= pos_["sl"]
                if htp_ or hsl_:
                    xp_ = pos_["tp"] if htp_ else pos_["sl"]
                    pnl_ = (xp_ - pos_["entry"]) / pos_["entry"]
            else:
                if lo_ < pos_["best"]: pos_["best"] = lo_
                if pos_["best"] <= pos_["entry"] - at_ * trail_act_:
                    pos_["sl"] = min(pos_["sl"], pos_["best"] + at_ * trail_dist_)
                htp_ = lo_ <= pos_["tp"]; hsl_ = hi_ >= pos_["sl"]
                if htp_ or hsl_:
                    xp_ = pos_["tp"] if htp_ else pos_["sl"]
                    pnl_ = (pos_["entry"] - xp_) / pos_["entry"]
            if htp_ or hsl_:
                dp_ = pnl_ * pos_["notional"] - pos_["notional"] * COMMISSION_PCT * 2
                eq_ += dp_
                trades_.append({"dollar_pnl": dp_, "result": "TP" if htp_ else "SL"})
                pos_ = None
        if pos_ is None:
            sig_ = "long" if bool(long_sig[ts_]) else ("short" if bool(short_sig[ts_]) else None)
            if sig_:
                rc_ = eq_ * RISK_PCT; qty_ = rc_ / sd_; not_ = qty_ * cl_
                sl_ = cl_ - sd_ if sig_ == "long" else cl_ + sd_
                tp_ = cl_ + at_ * tp_mult_ if sig_ == "long" else cl_ - at_ * tp_mult_
                pos_ = {"direction": sig_, "entry": cl_, "sl": sl_, "tp": tp_,
                        "best": cl_, "notional": not_}

    if not trades_: return {"ret": float("nan"), "pf": float("nan"), "wr": float("nan"), "n": 0}
    tdf_ = pd.DataFrame(trades_)
    w_   = tdf_[tdf_["dollar_pnl"] > 0]; l_ = tdf_[tdf_["dollar_pnl"] <= 0]
    ret_ = (eq_ / INITIAL_CAPITAL - 1) * 100
    gp_  = w_["dollar_pnl"].sum() if not w_.empty else 0
    gl_  = l_["dollar_pnl"].sum() if not l_.empty else 0
    pf_  = gp_ / abs(gl_) if gl_ != 0 else float("inf")
    wr_  = len(w_) / len(tdf_) * 100
    return {"ret": round(ret_, 2), "pf": round(pf_, 3), "wr": round(wr_, 1), "n": len(tdf_)}

# Build result grids
ret_grid = pd.DataFrame(index=pb_vals, columns=tp_vals, dtype=float)
pf_grid  = pd.DataFrame(index=pb_vals, columns=tp_vals, dtype=float)
n_grid   = pd.DataFrame(index=pb_vals, columns=tp_vals, dtype=float)

for pb in pb_vals:
    for tp in tp_vals:
        r = run_sim(df, pb, tp)
        ret_grid.loc[pb, tp] = r["ret"]
        pf_grid.loc[pb, tp]  = r["pf"]
        n_grid.loc[pb, tp]   = r["n"]

ret_grid.index.name  = "PB%"
pf_grid.index.name   = "PB%"

print("Return (%) grid  [PB% rows × TP mult cols]")
print(ret_grid.to_string())
print("\nProfit Factor grid")
print(pf_grid.to_string())
print("\nTrade Count grid")
print(n_grid.to_string())

# Heatmap visualisation
fig, (ax_ret, ax_pf) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f"Parameter Sweep  ·  {TICKER} {INTERVAL}  ·  ADX>{ADX2} +body filter", fontsize=12)

for grid, ax, title, fmt, cmap_ in [
    (ret_grid, ax_ret, "Return (%)",     "{:+.1f}%", "RdYlGn"),
    (pf_grid,  ax_pf,  "Profit Factor", "{:.2f}",   "RdYlGn"),
]:
    vals = grid.values.astype(float)
    finite = vals[~np.isnan(vals) & ~np.isinf(vals)]
    vmax   = np.percentile(np.abs(finite), 95) if len(finite) else 1
    im = ax.imshow(vals, cmap=cmap_, vmin=-vmax if title.startswith("Return") else 0,
                   vmax=vmax, aspect="auto")
    ax.set_xticks(range(len(tp_vals))); ax.set_xticklabels([f"TP×{t}" for t in tp_vals], fontsize=9)
    ax.set_yticks(range(len(pb_vals))); ax.set_yticklabels([f"PB={p}%" for p in pb_vals], fontsize=9)
    ax.set_title(title, fontsize=10)
    for r in range(len(pb_vals)):
        for c in range(len(tp_vals)):
            v = vals[r, c]
            txt = fmt.format(v) if not (np.isnan(v) or np.isinf(v)) else "—"
            ax.text(c, r, txt, ha="center", va="center", fontsize=8,
                    color="black" if abs(v) < vmax * 0.6 else "white")
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.tight_layout()
plt.savefig(f"backtest_apm_sweep_{TICKER.replace('-','').lower()}_{INTERVAL}.png", dpi=150, bbox_inches="tight")
plt.show()

# Best config by return
best_idx = ret_grid.stack().idxmax()
print(f"\nBest return: PB={best_idx[0]}%  TP×{best_idx[1]}"
      f"  →  {ret_grid.loc[best_idx[0], best_idx[1]]:+.2f}%"
      f"  PF={pf_grid.loc[best_idx[0], best_idx[1]]:.3f}"
      f"  Trades={int(n_grid.loc[best_idx[0], best_idx[1]])}")


Return (%) grid  [PB% rows × TP mult cols]
       1.0   1.5   2.0   2.5   3.0
PB%                               
0.10 -2.37  0.59  2.94  0.81  0.49
0.15 -1.95  2.71  5.92  2.30  2.32
0.20 -3.56  1.02  4.18  0.62  0.64
0.30 -5.11 -0.41  4.46  0.46  0.80
0.40 -5.63  0.27  5.44  2.07  3.10

Profit Factor grid
        1.0    1.5    2.0    2.5    3.0
PB%                                    
0.10  0.813  1.042  1.224  1.058  1.035
0.15  0.861  1.173  1.401  1.148  1.149
0.20  0.770  1.060  1.257  1.037  1.037
0.30  0.731  0.980  1.237  1.024  1.041
0.40  0.746  1.011  1.257  1.095  1.142

Trade Count grid
       1.0   1.5   2.0   2.5   3.0
0.10  34.0  32.0  31.0  31.0  31.0
0.15  40.0  38.0  37.0  37.0  37.0
0.20  41.0  39.0  38.0  38.0  38.0
0.30  49.0  46.0  44.0  44.0  44.0
0.40  58.0  54.0  51.0  51.0  51.0



Best return: PB=0.15%  TP×2.0  →  +5.92%  PF=1.401  Trades=37


## 12. APM v1.1 — Backtest with updated defaults

In [43]:
# ── 12. APM v1.1 simulation ────────────────────────────────────────────────────
# v1.1 parameters (all changes from pine script update)
V11_PB       = 0.15   # pullback tolerance %
V11_TP       = 2.0    # TP multiplier       (was 2.5 in v1.0, 1.5 in v2 tighter)
V11_ADX      = 25     # ADX threshold       (was 20)
V11_TACT     = 1.5    # trail activate mult (was 1.0)
V11_TDIST    = 0.8    # trail distance mult (unchanged)
V11_BODY     = 0.15   # min body / ATR      (was None in v1.0)

tol11 = V11_PB / 100.0

# Entry signals
body_v11    = (df["Close"] - df["Open"]).abs() / df["ATR"] >= V11_BODY
long_pb11   = (
    (df["Low"].shift(1) <= df["EMA_FAST"].shift(1) * (1 + tol11)) &
    (df["Close"] > df["EMA_FAST"]) &
    (df["Close"] > df["Open"]) & body_v11
)
short_pb11  = (
    (df["High"].shift(1) >= df["EMA_FAST"].shift(1) * (1 - tol11)) &
    (df["Close"] < df["EMA_FAST"]) &
    (df["Close"] < df["Open"]) & body_v11
)

long_sig11  = (
    long_pb11 &
    (df["Close"]    > df["EMA_SLOW"]) &
    (df["EMA_FAST"] > df["EMA_MID"]) &
    (df["RSI"] >= RSI_LO_L) & (df["RSI"] <= RSI_HI_L) &
    (df["Volume"] >= df["VOL_MA"] * VOL_MULT) &
    (df["ADX"] > V11_ADX) & ~df["IS_PANIC"]
) & TRADE_LONGS

short_sig11 = (
    short_pb11 &
    (df["Close"]    < df["EMA_SLOW"]) &
    (df["EMA_FAST"] < df["EMA_MID"]) &
    (df["RSI"] >= RSI_LO_S) & (df["RSI"] <= RSI_HI_S) &
    (df["Volume"] >= df["VOL_MA"] * VOL_MULT) &
    (df["ADX"] > V11_ADX) & ~df["IS_PANIC"]
) & TRADE_SHORTS

print(f"v1.1 Long  signals : {long_sig11.sum():>4}")
print(f"v1.1 Short signals : {short_sig11.sum():>4}")
print(f"v1.1 Total         : {long_sig11.sum() + short_sig11.sum():>4}")

# ── Simulation loop ───────────────────────────────────────────────────────────
eq11      = INITIAL_CAPITAL
pos11     = None
trades11  = []
eqcurve11 = []

for ts, row in df.iterrows():
    close = float(row["Close"]); high  = float(row["High"])
    low   = float(row["Low"]);   atr   = float(row["ATR"])
    sd    = atr * SL_MULT
    htp11 = hsl11 = False; pnl11 = 0.0

    if pos11 is not None:
        d = pos11["direction"]
        if d == "long":
            if high > pos11["best"]: pos11["best"] = high
            if pos11["best"] >= pos11["entry"] + atr * V11_TACT:
                pos11["sl"] = max(pos11["sl"], pos11["best"] - atr * V11_TDIST)
            htp11 = high >= pos11["tp"];  hsl11 = low <= pos11["sl"]
            if htp11 or hsl11:
                xp11 = pos11["tp"] if htp11 else pos11["sl"]
                pnl11 = (xp11 - pos11["entry"]) / pos11["entry"]
        else:
            if low < pos11["best"]: pos11["best"] = low
            if pos11["best"] <= pos11["entry"] - atr * V11_TACT:
                pos11["sl"] = min(pos11["sl"], pos11["best"] + atr * V11_TDIST)
            htp11 = low <= pos11["tp"];  hsl11 = high >= pos11["sl"]
            if htp11 or hsl11:
                xp11 = pos11["tp"] if htp11 else pos11["sl"]
                pnl11 = (pos11["entry"] - xp11) / pos11["entry"]

        if htp11 or hsl11:
            dp11 = pnl11 * pos11["notional"] - pos11["notional"] * COMMISSION_PCT * 2
            eq11 += dp11
            trades11.append({
                "direction": d, "entry_time": pos11["entry_time"], "exit_time": ts,
                "entry": pos11["entry"], "exit": xp11,
                "result": "TP" if htp11 else "SL",
                "pnl_pct": round(pnl11*100, 3),
                "dollar_pnl": round(dp11, 2), "equity": round(eq11, 2),
            })
            pos11 = None

    if pos11 is None:
        sig11 = "long" if bool(long_sig11[ts]) else ("short" if bool(short_sig11[ts]) else None)
        if sig11:
            rc   = eq11 * RISK_PCT; qty11 = rc / sd; not11 = qty11 * close
            sl11 = close - sd if sig11 == "long" else close + sd
            tp11 = close + atr * V11_TP if sig11 == "long" else close - atr * V11_TP
            pos11 = {"direction": sig11, "entry": close, "entry_time": ts,
                     "sl": sl11, "tp": tp11, "best": close, "notional": not11}

    eqcurve11.append({"time": ts, "equity": eq11})

t11df = pd.DataFrame(trades11)
print(f"\nv1.1 trades executed: {len(t11df)}")

# ── Stats ─────────────────────────────────────────────────────────────────────
print()
stats_block(tdf,    equity_curve, f"APM v1.0 defaults   (PB={PB_PCT}%  TP×{TP_MULT}  ADX>{ADX_THRESH}  no body)")
print()
stats_block(t2df,   eqcurve2,    f"APM v2  tighter     (PB={PB2}%  TP×{TP2}   ADX>{ADX2}  +body)")
print()
stats_block(t11df,  eqcurve11,   f"APM v1.1 new default (PB={V11_PB}%  TP×{V11_TP}   ADX>{V11_ADX}  +body)")

# ── Equity curve 3-way ────────────────────────────────────────────────────────
ec11 = pd.DataFrame(eqcurve11).set_index("time")["equity"]

plt.style.use("dark_background")
fig, axes = plt.subplots(2, 1, figsize=(18, 10),
                          gridspec_kw={"height_ratios": [3, 1]})
fig.suptitle(f"APM  ·  {TICKER} {INTERVAL}  ·  v1.0 vs v2-tighter vs v1.1", fontsize=13)

ax = axes[0]
ax.plot(ec1.index,  ec1.values,  color="#fc8181", lw=1.2, alpha=0.8,
        label=f"v1.0 defaults  Return: {(ec1.iloc[-1]/INITIAL_CAPITAL-1)*100:+.2f}%")
ax.plot(ec2.index,  ec2.values,  color="#f6ad55", lw=1.2, alpha=0.8,
        label=f"v2  tighter    Return: {(ec2.iloc[-1]/INITIAL_CAPITAL-1)*100:+.2f}%")
ax.plot(ec11.index, ec11.values, color="#68d391", lw=2.0,
        label=f"v1.1 new       Return: {(ec11.iloc[-1]/INITIAL_CAPITAL-1)*100:+.2f}%")
ax.axhline(INITIAL_CAPITAL, color="white", ls=":", lw=0.8, alpha=0.4)
ax.set_ylabel("Equity ($)")
ax.legend(loc="lower left", fontsize=9)
ax.grid(alpha=0.15)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))

# Drawdown
def rolling_dd(ec):
    mx = ec.cummax()
    return (ec - mx) / mx * 100

ax2 = axes[1]
ax2.fill_between(ec1.index,  rolling_dd(ec1).values,  0, alpha=0.35, color="#fc8181", label="v1.0 DD")
ax2.fill_between(ec2.index,  rolling_dd(ec2).values,  0, alpha=0.35, color="#f6ad55", label="v2  DD")
ax2.fill_between(ec11.index, rolling_dd(ec11).values, 0, alpha=0.45, color="#68d391", label="v1.1 DD")
ax2.set_ylabel("Drawdown (%)")
ax2.legend(loc="lower left", fontsize=8)
ax2.grid(alpha=0.15)
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))

plt.tight_layout()
v11_png = f"backtest_apm_v11_{TICKER.replace('-','').lower()}_{INTERVAL}.png"
plt.savefig(v11_png, dpi=150, bbox_inches="tight")
plt.show()
print(f"Chart saved → {v11_png}")

# ── Per-trade bar chart ───────────────────────────────────────────────────────
if not t11df.empty:
    fig2, ax_bars = plt.subplots(figsize=(18, 4))
    fig2.suptitle(f"APM v1.1 · Per-trade P&L  ·  {TICKER} {INTERVAL}", fontsize=12)
    bar_c11 = ["#68d391" if v >= 0 else "#fc8181" for v in t11df["dollar_pnl"]]
    ax_bars.bar(range(len(t11df)), t11df["dollar_pnl"], color=bar_c11, width=0.6)
    ax_bars.axhline(0, color="white", lw=0.7, alpha=0.5)
    ax_bars.set_xticks(range(len(t11df)))
    ax_bars.set_xticklabels([f"T{i+1}\n{r}" for i, r in enumerate(t11df["result"])], fontsize=7)
    ax_bars.set_ylabel("P&L ($)")
    ax_bars.grid(alpha=0.15)
    plt.tight_layout()
    plt.savefig(f"backtest_apm_v11_trades_{TICKER.replace('-','').lower()}_{INTERVAL}.png", dpi=150, bbox_inches="tight")
    plt.show()


v1.1 Long  signals :   12
v1.1 Short signals :   31
v1.1 Total         :   43

v1.1 trades executed: 37

  APM v1.0 defaults   (PB=0.3%  TP×2.5  ADX>20  no body)
  Return        :    -20.32 %   Max DD: -20.75 %
  Profit factor :      0.39   Trades: 74
  Win rate      :     48.6 %   TP exits: 5 / 74
  Avg win  $    : $   +36.77   Avg loss $: $   -88.29   R:R: 0.42

  APM v2  tighter     (PB=0.15%  TP×1.5   ADX>25  +body)
  Return        :     +2.71 %   Max DD: -4.44 %
  Profit factor :      1.17   Trades: 38
  Win rate      :     65.8 %   TP exits: 23 / 38
  Avg win  $    : $   +73.32   Avg loss $: $  -120.16   R:R: 0.61

  APM v1.1 new default (PB=0.15%  TP×2.0   ADX>25  +body)
  Return        :     +7.03 %   Max DD: -4.30 %
  Profit factor :      1.47   Trades: 37
  Win rate      :     67.6 %   TP exits: 16 / 37
  Avg win  $    : $   +87.58   Avg loss $: $  -123.88   R:R: 0.71


Chart saved → backtest_apm_v11_btcusd_30m.png


In [44]:

# ── 13. Scaled projection: $2,000 starting balance ───────────────────────────
SCALE_CAP = 2_000.0

# Exact copy of v1.1 loop, only INITIAL_CAPITAL swapped → SCALE_CAP
eq_sc      = SCALE_CAP
pos_sc     = None
trades_sc  = []
eqcurve_sc = []

for ts, row in df.iterrows():
    close = float(row["Close"]); high  = float(row["High"])
    low   = float(row["Low"]);   atr   = float(row["ATR"])
    sd    = atr * SL_MULT          # stop distance (same as original)
    htp_sc = hsl_sc = False; pnl_sc = 0.0

    if pos_sc is not None:
        d = pos_sc["direction"]
        if d == "long":
            if high > pos_sc["best"]: pos_sc["best"] = high
            if pos_sc["best"] >= pos_sc["entry"] + atr * V11_TACT:
                pos_sc["sl"] = max(pos_sc["sl"], pos_sc["best"] - atr * V11_TDIST)
            htp_sc = high >= pos_sc["tp"];  hsl_sc = low <= pos_sc["sl"]
            if htp_sc or hsl_sc:
                xp_sc  = pos_sc["tp"] if htp_sc else pos_sc["sl"]
                pnl_sc = (xp_sc - pos_sc["entry"]) / pos_sc["entry"]
        else:
            if low < pos_sc["best"]: pos_sc["best"] = low
            if pos_sc["best"] <= pos_sc["entry"] - atr * V11_TACT:
                pos_sc["sl"] = min(pos_sc["sl"], pos_sc["best"] + atr * V11_TDIST)
            htp_sc = low <= pos_sc["tp"];  hsl_sc = high >= pos_sc["sl"]
            if htp_sc or hsl_sc:
                xp_sc  = pos_sc["tp"] if htp_sc else pos_sc["sl"]
                pnl_sc = (pos_sc["entry"] - xp_sc) / pos_sc["entry"]

        if htp_sc or hsl_sc:
            dp_sc  = pnl_sc * pos_sc["notional"] - pos_sc["notional"] * COMMISSION_PCT * 2
            eq_sc += dp_sc
            trades_sc.append({"result": "TP" if htp_sc else "SL", "dollar_pnl": dp_sc})
            pos_sc = None

    if pos_sc is None:
        sig_sc = "long" if bool(long_sig11[ts]) else ("short" if bool(short_sig11[ts]) else None)
        if sig_sc:
            rc_sc   = eq_sc * RISK_PCT
            qty_sc  = rc_sc / sd
            not_sc  = qty_sc * close
            sl_sc   = close - sd if sig_sc == "long" else close + sd
            tp_sc   = close + atr * V11_TP if sig_sc == "long" else close - atr * V11_TP
            pos_sc  = {"direction": sig_sc, "entry": close, "sl": sl_sc,
                       "tp": tp_sc, "best": close, "notional": not_sc}

    eqcurve_sc.append({"time": ts, "equity": eq_sc})

# ── Results ───────────────────────────────────────────────────────────────────
profit    = eq_sc - SCALE_CAP
ret_sc    = profit / SCALE_CAP * 100
eq_vals   = [e["equity"] for e in eqcurve_sc]
mdd_sc, peak_sc = 0.0, SCALE_CAP
for v in eq_vals:
    if v > peak_sc: peak_sc = v
    dd = (v - peak_sc) / peak_sc * 100
    if dd < mdd_sc: mdd_sc = dd

wins_sc   = [t["dollar_pnl"] for t in trades_sc if t["dollar_pnl"] > 0]
losses_sc = [t["dollar_pnl"] for t in trades_sc if t["dollar_pnl"] <= 0]
tp_sc_cnt = sum(1 for t in trades_sc if t["result"] == "TP")

print("=" * 52)
print(f"  APM v1.1  |  Starting capital : ${SCALE_CAP:,.0f}")
print("=" * 52)
print(f"  Total profit  : ${profit:+,.2f}")
print(f"  Final balance : ${eq_sc:,.2f}")
print(f"  Return        : {ret_sc:+.2f} %")
print(f"  Max drawdown  : {mdd_sc:.2f} %")
print(f"  Trades        : {len(trades_sc)}")
print(f"  Win rate      : {len(wins_sc)/len(trades_sc)*100:.1f} %")
print(f"  TP exits      : {tp_sc_cnt} / {len(trades_sc)}")
if wins_sc:   print(f"  Avg win       : ${sum(wins_sc)/len(wins_sc):+.2f}")
if losses_sc: print(f"  Avg loss      : ${sum(losses_sc)/len(losses_sc):+.2f}")
print("=" * 52)
print(f"\n  (Comparison: $10k run → +7.03%, Max DD -4.30%)")


  APM v1.1  |  Starting capital : $2,000
  Total profit  : $+140.59
  Final balance : $2,140.59
  Return        : +7.03 %
  Max drawdown  : -4.30 %
  Trades        : 37
  Win rate      : 67.6 %
  TP exits      : 16 / 37
  Avg win       : $+17.52
  Avg loss      : $-24.78

  (Comparison: $10k run → +7.03%, Max DD -4.30%)
